In [0]:

from pyspark.sql import functions as F

# ============================================================
# 1. CONFIGURATION
# ============================================================

bronze_table = "retail_ops.bronze.orders_raw"
silver_table = "retail_ops.silver.orders_clean"

# ============================================================
# 2. READ BRONZE DATA
# ============================================================

raw_df = spark.table(bronze_table)

print("Bronze row count:", raw_df.count())
print("Original columns:", raw_df.columns)

# ============================================================
# 3. STANDARDIZE COLUMN NAMES
# ============================================================

df = raw_df.toDF(*[
    c.strip().replace(" ", "_").lower()
    for c in raw_df.columns
])

# ============================================================
# 4. CLEAN AND CONVERT DATA TYPES
# ============================================================

df = (
    df
    .withColumn(
        "quantity",
        F.expr("try_cast(quantity AS INT)")
    )
    .withColumn(
        "unitprice",
        F.expr("try_cast(unitprice AS DOUBLE)")
    )
    .withColumn(
        "invoicedate",
        F.coalesce(
            F.to_timestamp("invoicedate", "M/d/yyyy H:mm:ss"),
            F.to_timestamp("invoicedate", "d/M/yyyy H:mm:ss")
        )
    )
    .withColumn(
        "customerid",
        F.expr("try_cast(customerid AS BIGINT)")
    )
)

# ============================================================
# 5. CALCULATE REVENUE
# ============================================================

df = df.withColumn(
    "revenue",
    F.round(F.col("quantity") * F.col("unitprice"), 2)
)

# ============================================================
# 6. DATA QUALITY CHECKS
# ============================================================

df = df.withColumn(
    "dq_issue",
    F.when(F.col("invoiceno").isNull(), "missing_invoice")
     .when(F.col("stockcode").isNull(), "missing_stockcode")
     .when(F.col("quantity").isNull(), "invalid_quantity")
     .when(F.col("unitprice").isNull(), "invalid_unitprice")
     .when(F.col("invoicedate").isNull(), "invalid_invoicedate")
)

print("Data quality issue summary:")

display(
    df.groupBy("dq_issue")
      .count()
      .orderBy(F.col("count").desc())
)

# ============================================================
# 7. FILTER VALID SALES TRANSACTIONS
# ============================================================

silver_df = df.filter(
    F.col("dq_issue").isNull()
    & (F.col("quantity") > 0)
    & (F.col("unitprice") >= 0)
    & (~F.upper(F.trim(F.col("invoiceno"))).startswith("C"))
)

# ============================================================
# 8. SAVE SILVER DELTA TABLE
# ============================================================

(
    silver_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(silver_table)
)

# ============================================================
# 9. VALIDATE RESULTS
# ============================================================

print("Silver table created:", silver_table)
print("Silver row count:", spark.table(silver_table).count())

display(
    spark.table(silver_table)
    .select(
        "invoiceno",
        "stockcode",
        "description",
        "quantity",
        "invoicedate",
        "unitprice",
        "customerid",
        "country",
        "revenue"
    )
    .limit(10)
)
